# DQL Analytical Functions: ROW_NUMBER, RANK, and DENSE_RANK

Ranking functions assign positions to rows inside a window.


## How to Think About Analytical Functions

Analytical functions use windows to compare each row with related rows while keeping row-level detail. This is different from `GROUP BY`, which collapses rows into summaries.

A window usually defines:

* `partitionBy`, which chooses the group of related rows.
* `orderBy`, which chooses the row sequence inside each group.
* An optional frame, which controls how many rows are visible to the function.

Use analytical functions for ranking, top-N logic, previous or next row comparisons, and first or last values within a group.

## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Window Setup

A window defines the rows each analytical function can see. This example partitions by department and orders by salary.

Analytical functions use windows. A window tells Spark which nearby or related rows each row can compare itself to.

Two important window ideas:

* `partitionBy` splits rows into groups, like departments.
* `orderBy` decides the row order inside each group.


In [ ]:
dept_salary_window = Window.partitionBy("deptno").orderBy(F.col("sal").desc())
all_salary_window = Window.orderBy(F.col("sal").desc())


## ROW_NUMBER, RANK, and DENSE_RANK

`row_number` gives every row a unique sequence. `rank` leaves gaps after ties. `dense_rank` does not leave gaps.

These functions assign positions to rows.

Use this difference:

* `row_number` always gives unique numbers.
* `rank` gives the same rank to ties but leaves gaps.
* `dense_rank` gives the same rank to ties and does not leave gaps.


In [ ]:
emp.select(
    "deptno", "empno", "ename", "job", "sal",
    F.row_number().over(dept_salary_window).alias("row_number_in_dept"),
    F.rank().over(dept_salary_window).alias("rank_in_dept"),
    F.dense_rank().over(dept_salary_window).alias("dense_rank_in_dept")
).orderBy("deptno", "row_number_in_dept").show(30)

spark.sql("""
SELECT deptno, empno, ename, job, sal,
       ROW_NUMBER() OVER (PARTITION BY deptno ORDER BY sal DESC) AS row_number_in_dept,
       RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS rank_in_dept,
       DENSE_RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS dense_rank_in_dept
FROM emp
ORDER BY deptno, row_number_in_dept
""").show(30)
